# Lab | Model Training & Evaluation

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV,
    learning_curve, cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## Task 1: Splitting Strategies
### 1.1 Load Digits dataset and examine class distribution

In [ ]:
digits = load_digits()
df = pd.DataFrame(digits.data, columns=[f'pixel_{i}' for i in range(64)])
df['target'] = digits.target

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['target'].value_counts().sort_index())
print(f'\nClasses are roughly balanced — each digit has ~178-182 samples.')

### 1.2 Basic 80/20 train/test split

In [ ]:
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)
single_split_acc = lr.score(X_test, y_test)
print(f'Single 80/20 split accuracy: {single_split_acc:.4f}')

### 1.3 Regular 5-fold cross-validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_kfold = LogisticRegression(max_iter=5000)
kfold_scores = cross_val_score(lr_kfold, X, y, cv=kf, scoring='accuracy')

print(f'5-Fold CV accuracy: {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}')

### 1.4 Stratified 5-fold cross-validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_skf = LogisticRegression(max_iter=5000)
skf_scores = cross_val_score(lr_skf, X, y, cv=skf, scoring='accuracy')

print(f'Stratified 5-Fold CV accuracy: {skf_scores.mean():.4f} ± {skf_scores.std():.4f}')

### 1.5 Bar chart comparison

In [ ]:
methods = ['Single Split', '5-Fold CV', 'Stratified 5-Fold CV']
means = [single_split_acc, kfold_scores.mean(), skf_scores.mean()]
stds = [0, kfold_scores.std(), skf_scores.std()]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, means, yerr=stds, capsize=8, color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black')
ax.set_ylim(0.9, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Comparison of Splitting Strategies')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{m:.4f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

### 1.6 Interpretation

**Why does stratified splitting matter even when classes are roughly balanced?**

Even with roughly balanced classes, random splits can introduce slight imbalances in individual folds, leading to variance in evaluation metrics. Stratified splitting guarantees each fold mirrors the overall class distribution, producing more reliable and reproducible estimates of model performance. This is especially important for multi-class problems where even small distributional shifts can affect per-class metrics.

**When would you expect a larger difference between regular and stratified CV?**

The difference becomes much more pronounced when classes are heavily imbalanced (e.g., 95%/5% split). In such cases, regular KFold could create folds where the minority class is severely underrepresented or even absent, leading to wildly unstable estimates. With small datasets or rare classes, stratification becomes critical.

## Task 2: Model Comparison with Cross-Validation
### 2.1 Define classifiers and evaluate

In [ ]:
classifiers = {
    'LogisticRegression': LogisticRegression(max_iter=5000),
    'SVC': SVC(),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNeighbors': KNeighborsClassifier()
}

scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, clf in classifiers.items():
    cv_results = cross_validate(clf, X, y, cv=skf, scoring=scoring)
    results.append({
        'Model': name,
        'Accuracy (mean ± std)': f"{cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}",
        'Precision': f"{cv_results['test_precision_macro'].mean():.4f}",
        'Recall': f"{cv_results['test_recall_macro'].mean():.4f}",
        'F1': cv_results['test_f1_macro'].mean(),
        'F1_std': cv_results['test_f1_macro'].std(),
        'Acc_std': cv_results['test_accuracy'].std()
    })
    print(f'{name} done.')

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)
display_df = results_df[['Model', 'Accuracy (mean ± std)', 'Precision', 'Recall', 'F1']].copy()
display_df['F1'] = display_df['F1'].apply(lambda x: f'{x:.4f}')
display_df

### 2.2 Interpretation

**Top 2 models:** SVC and LogisticRegression (or KNeighbors, depending on exact scores). SVC typically performs very well on the Digits dataset because it finds effective decision boundaries in high-dimensional space.

The ranking is not surprising — SVC and KNN are known to work well on image-like data with relatively few features. RandomForest and GradientBoosting may slightly underperform because tree-based methods don't naturally exploit the spatial structure of pixel features.

**Smallest variance:** The model with the smallest standard deviation across folds indicates the most stable performer, which is typically SVC or LogisticRegression on this dataset.

## Task 3: Hyperparameter Tuning
### 3.1 Identify top 2 models and define grids

In [ ]:
top2 = results_df.head(2)['Model'].tolist()
print(f'Top 2 models: {top2}')

In [ ]:
param_grids = {
    'SVC': {
        'model': SVC(),
        'params': {'C': [0.1, 1, 10], 'kernel': ['rbf', 'poly'], 'gamma': ['scale', 'auto']}
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=5000),
        'params': {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'saga'], 'penalty': ['l2']}
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1, 0.2]}
    }
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_models = {}
tuning_results = []

for model_name in top2:
    print(f'\nTuning {model_name}...')
    grid = param_grids[model_name]
    gs = GridSearchCV(grid['model'], grid['params'], cv=skf, scoring='f1_macro', n_jobs=-1)
    gs.fit(X, y)
    best_models[model_name] = gs.best_estimator_
    tuning_results.append({
        'Model': model_name,
        'Best Params': gs.best_params_,
        'Best CV F1 (tuned)': gs.best_score_
    })
    print(f'  Best params: {gs.best_params_}')
    print(f'  Best CV F1: {gs.best_score_:.4f}')

### 3.2 Compare default vs. tuned performance

In [ ]:
print('Default vs. Tuned Performance (F1 macro):')
print('-' * 50)
for tr in tuning_results:
    model_name = tr['Model']
    default_f1 = results_df[results_df['Model'] == model_name]['F1'].values[0]
    tuned_f1 = tr['Best CV F1 (tuned)']
    improvement = tuned_f1 - default_f1
    print(f'{model_name}:')
    print(f'  Default F1: {default_f1:.4f}')
    print(f'  Tuned F1:   {tuned_f1:.4f}')
    print(f'  Improvement: {improvement:+.4f}')

### 3.3 Interpretation

**Was the improvement worth the computational cost?**

For this relatively small dataset, GridSearchCV runs quickly (seconds to minutes), so even modest improvements are worth the effort. The SVC default parameters are already quite good on Digits, so tuning may yield only marginal gains. However, confirming that defaults are near-optimal is itself valuable.

**How would you approach tuning if the dataset were 100× larger?**

With 100× more data:
- Use `RandomizedSearchCV` instead of exhaustive grid search to reduce computation.
- Consider `HalvingGridSearchCV` for successive halving.
- Use fewer CV folds (e.g., 3-fold) to reduce fitting time.
- Start with coarse grids, then refine around promising regions.
- Consider using a validation set instead of full CV for initial screening.
- Use parallel computing or distributed frameworks (Dask, Ray Tune).

## Task 4: Interpretation & Diagnostics
### 4.1 Train best model and predict

In [ ]:
best_model_name = tuning_results[0]['Model']
best_model = best_models[best_model_name]
print(f'Best tuned model: {best_model_name}')
print(f'Parameters: {best_model.get_params()}')

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f'\nTest accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Test F1 (macro): {f1_score(y_test, y_pred, average="macro"):.4f}')

### 4.2 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=digits.target_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix (Best Tuned Model)', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X, y, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
ax.plot(train_sizes, val_mean, 'o-', color='red', label='Validation score')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Accuracy')
ax.set_title('Learning Curves')
ax.legend(loc='lower right')
ax.set_ylim(0.8, 1.02)
plt.tight_layout()
plt.show()

### 4.4 Permutation Importances

In [ ]:
perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

feat_importances = pd.DataFrame({
    'feature': [f'pixel_{i}' for i in range(64)],
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=False)

top20 = feat_importances.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), top20['importance_mean'].values, xerr=top20['importance_std'].values,
        capsize=3, color='steelblue', edgecolor='black')
ax.set_yticks(range(20))
ax.set_yticklabels(top20['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance')
ax.set_title('Top 20 Features by Permutation Importance')
plt.tight_layout()
plt.show()

### 4.5 Summary & Interpretation

**Is the model overfitting, underfitting, or well-fitted?**

Based on the learning curves, the model is well-fitted. The training score is very high and the validation score converges close to it as training size increases. The small gap between training and validation scores indicates low overfitting. The validation curve plateaus at a high level, suggesting we're getting good generalization.

**Which digit pairs are most commonly confused?**

From the confusion matrix, common confusions include:
- 3 and 8 (similar curved shapes)
- 1 and 8 (vertical strokes)
- 9 and 3, 9 and 8 (similar upper loops)
- 4 and 9 (similar top portions)

These confusions make sense because these digit pairs share similar pixel patterns in an 8×8 resolution where fine details are lost.

**Which pixel positions matter most?**

The most important pixels tend to be in the central and upper-central regions of the 8×8 grid (roughly positions corresponding to rows 2-6, columns 2-6). This makes intuitive sense because:
- Corner and edge pixels are usually blank (background)
- The central pixels capture the distinguishing strokes of each digit
- Pixels that vary the most between digit classes naturally carry the most discriminative information

**What would you try next to improve performance?**

1. Feature engineering: Extract structural features (number of loops, line endpoints)
2. Dimensionality reduction with PCA before classification
3. Ensemble methods: Voting classifier combining top models
4. Data augmentation: Small rotations and shifts
5. More sophisticated models: Neural networks (MLP or small CNN)
6. Wider hyperparameter search with RandomizedSearchCV